# Chapter 01 — Beyond the Chat Box

**Companion to Applied AI**

Question: What actually changes when a model call becomes a recorded operation?

By the end of this notebook you will have:

- called a mock model as chat (a string) and as an operation (a record)
- processed the structured result with deterministic code
- seen why a string forces fragile re-parsing

## What this notebook demonstrates
This is a synthetic experiment. A tiny mock stands in for a model so the notebook runs with no API key. Nothing here measures a real model.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)
import time, uuid

seed: 42


## 1. The chat-shaped call: a string arrives
A person must read it before software can act on it.

In [2]:
def chat_generate(prompt: str) -> str:
    """Mock chat endpoint: returns only prose."""
    return ("Review of payments.csv: I found 2 errors. "
            "Row 14 has code E_OVERLIMIT and row 31 has code E_MISSING_REF. "
            "Overall it looks mostly fine.")

reply = chat_generate("review payments.csv")
print(reply)
print("type:", type(reply).__name__)

Review of payments.csv: I found 2 errors. Row 14 has code E_OVERLIMIT and row 31 has code E_MISSING_REF. Overall it looks mostly fine.
type: str


## 2. The operation-shaped call: a record arrives
Same underlying judgment, but returned as fields software can inspect, plus an identity for the record.

In [3]:
def operate_review(task_id: str) -> dict:
    """Mock operation endpoint: structured result + record identity."""
    return {
        "task_id": task_id,
        "call_id": "call-" + uuid.uuid4().hex[:8],
        "timestamp": time.time(),
        "errors": [
            {"row": 14, "code": "E_OVERLIMIT"},
            {"row": 31, "code": "E_MISSING_REF"},
        ],
        "verdict": "needs_fix",
    }

op = operate_review("task-payments-001")
print(op)
assert isinstance(op["errors"], list)
assert op["verdict"] in ("ok", "needs_fix")

{'task_id': 'task-payments-001', 'call_id': 'call-b857921b', 'timestamp': 1789397522.6765916, 'errors': [{'row': 14, 'code': 'E_OVERLIMIT'}, {'row': 31, 'code': 'E_MISSING_REF'}], 'verdict': 'needs_fix'}


## 3. Downstream code can act without re-reading prose

In [4]:
overlimit_rows = [e["row"] for e in op["errors"] if e["code"] == "E_OVERLIMIT"]
print("rows to block:", overlimit_rows)
assert overlimit_rows == [14]

rows to block: [14]


## 4. Break it deliberately: re-parse the chat string
A reworded reply breaks the fragile parser. The structured record is unaffected.

In [5]:
import re
def fragile_parse(reply: str):
    return [int(x) for x in re.findall(r"Row (\d+)", reply)]

print("original reply parses as:", fragile_parse(reply))
reworded = "Review done: a couple of issues, one around row fourteen, another at row 31."
print("reworded reply parses as:", fragile_parse(reworded), "<- silently wrong")
assert fragile_parse(reworded) != [14, 31]

original reply parses as: [14]
reworded reply parses as: [] <- silently wrong


## Interpretation
- Supports: returning fields instead of prose lets deterministic code verify, count, and route.
- Does NOT support: any claim about real model quality; the mock is fixed.

## Try it yourself
1. Add a third error to both paths and rerun.
2. Change the reworded reply and watch the parser fail differently.
3. Record a second operation and confirm `call_id` values differ while `task_id` stays put.